# Run a simulation

Runs **one simulation** with a live dashboard that refreshes every `UPDATE_EVERY_DAYS` simulated days:
the landscape now, change since the start, trail traffic by role, and the swarm and environment metrics so far.
The run is recorded in `RUN_DIR` (see `beaversim/records.py`); open `notebooks/analysis/run_report.ipynb` on it
for the full report.

In [ ]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "beaversim").is_dir())
sys.path.insert(0, str(root))

from beaversim import viz
from beaversim.experiments import run, run_montecarlo, apply_overrides

## Parameters

In [ ]:
RUN_DIR = root / "output/simulations/montana_demo/seed_1"   # where the run record is written
UPDATE_EVERY_DAYS = 3        # dashboard refresh interval (each refresh costs ~0.5 s)
MEDIUM = "chapter"           # "chapter", "paper" or "slides"

## Configuration

In [ ]:
SITE = root / "output/RumneyRanch_Montana/RGB/A2/raw"     # prepared site (data_acquisition pipeline output)

config = {
    "backend_type": "beavers_visualizer",
    "environment": {
        "name": "beavers_environment",
        "elevation_file_path": str(SITE / "baseline_matrix.npy"),
        "metadata_file_path": str(SITE / "processing_metadata.json"),
        # Vegetation
        "vegetation_quality_range": [0, 10],
        "grass_growth_interval": [0, 8],
        "grass_growth_rate": 1e-4,
        "grass_growth_sigma": 0.5,
        "mean_reversion_strength": 0.0005,
        # Stigmergy
        "visits_reset": 0.99,
        # Water
        "flow_info": {
            "river_growth_velocity": 1e-4,
            "river_growth_interval": [-3, 0],
            "streams_depth": 10,
            "direction": [1, 1],
            "strength": 0.2,
        },
        "print": False,
    },
    "robot": {
        "home_base_position": [[60, 60]],                  # metres; one or more lodges
        "role": ["explorer", "builder"],                   # each agent draws one at random
        # Per-role parameters: [explorer, builder]
        "maximum_load": [1000, 10],
        "harvest_interval": [[6, 9], [0, 2]],
        "epsilon_greedy": [0.8, 1.0],
        "exploration_map": ["vegetation_quality", "vegetation_visits"],
        "vegetation_removal": [3.0, 1.0],
        # Motion
        "controller": {
            "name": "P_repulsive",
            "Kp": 0.5, "Kd": 2.0, "Ki": 1e-5,
            "neighbourhood_size": 2,
            "accuracy": 0.5,
            "map_repulsive": "vegetation_quality",
            "beta_repulsive": 0.4,
        },
        "dynamics": {"name": "integrator", "mass": 5, "friction": 1},
        # Stigmergy and adaptation
        "visit_increase": 1.0,
        "decay_values": [0.001, 0.001],                    # [maximum_load decay, harvest_interval widening]
        "print": False,
    },
    "simulation": {
        "seed": 1,
        "timedelta": 1.0,                                  # hours per step
        "schedule_policy": "sequential",
        "gui": False,
        "number_of_agents": 10,
        "number_of_steps": 180,                            # duration in DAYS
        "downsampling": 15,                                # days between saved snapshots
        "save_path": None,                                 # unused: the output folder is set below
        "print": False,
    },
}

### Configuration reference

**environment**
- `elevation_file_path`, `metadata_file_path`: prepared site; the metadata gives `target_resolution_m`.
- `vegetation_quality_range`: `[min, max]`; the map is rescaled to `[-streams_depth, max]` (negative = water).
- `grass_growth_interval`: values where vegetation regrows. `grass_growth_rate`, `grass_growth_sigma`,
  `mean_reversion_strength`: seasonal growth, noise, pull towards the seasonal target.
- `visits_reset`: per-step decay of the visit ledger (stigmergic memory).
- `flow_info`: `streams_depth` (water depth scale), `direction` / `strength` (drift applied to agents in water),
  `river_growth_velocity` / `river_growth_interval` (slow deepening of channels).

**robot** (lists with two entries are `[explorer, builder]`)
- `home_base_position`: lodge(s) in metres. `role`: roles drawn at random per agent.
- `maximum_load`: load that triggers a return home. `harvest_interval`: vegetation range an agent harvests.
- `epsilon_greedy`: probability of harvesting when in range. `vegetation_removal`: amount removed per harvest.
- `exploration_map`: what the exploration utility uses, `vegetation_quality` or `vegetation_visits`.
- `controller`: PID gains, `accuracy` (waypoint tolerance, cells), repulsive field (`map_repulsive`,
  `neighbourhood_size`, `beta_repulsive` = weight of repulsion vs attraction).
- `dynamics`: double integrator with `mass` and `friction`.
- `visit_increase`: ledger increment per visit. `decay_values`: adaptation of maximum load and harvest interval.

**simulation**
- `seed`: controls every random draw. `timedelta`: hours per step. `number_of_steps`: duration in **days**.
- `downsampling`: days between saved snapshots. `number_of_agents`.

## Run
The dashboard below updates in place while the simulation runs. Interrupting the kernel stops the run;
nothing is written until the run completes.

In [ ]:
viz.use_theme(MEDIUM)
run_dir = run(config, RUN_DIR, on_update=viz.live.LiveDashboard(), update_every_days=UPDATE_EVERY_DAYS)
print("Run recorded in", run_dir)

## Next
Open `notebooks/analysis/run_report.ipynb` and set `RUN_DIR` to the folder printed above.